# Phase 29: Thread 3, Step 2: Topology at Scale (2.5x pool)

Same persistent-homology featurizer as Phase 6b (byte-for-byte, spliced
verbatim -- alpha complexes on 12A neighborhoods, H0/H1/H2, Betti radii
2/3/5A, burial features, rotation-invariance re-verified below), just
pointed at Phase 28's ~15,000-entry pool instead of Phase 6a's 6,000. CPU
only (gudhi has no GPU path) -- expect roughly 2.5x Phase 6b's 66 minutes.

## Cell 0: Install dependencies

In [ ]:
!pip install -q torch-geometric==2.6.1 gudhi==3.10.1 biopython==1.84

## Cell 1: Load Phase 28's pool and re-acquire heavy atoms

In [ ]:
import gzip, os, time, urllib.request, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import gudhi
from scipy.spatial import cKDTree
from Bio.PDB import MMCIFParser, PDBParser

warnings.filterwarnings("ignore")


def locate(fname):
    c = []
    for root in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if root.exists():
            c.extend(root.rglob(fname))
    if not c:
        raise FileNotFoundError(fname)
    return max(c, key=lambda p: p.stat().st_size)


blob = torch.load(locate("phase28_graphs.pt"), map_location="cpu", weights_only=False)
dataset = blob["dataset"]
print(f"chains {len(dataset):,} | residues {sum(d.num_nodes for d in dataset):,} "
      f"| clusters {len({d.cluster for d in dataset}):,}")

RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)
needed = sorted({d.pdb_id for d in dataset})
print(f"structures to re-fetch: {len(needed):,}")


def fetch(pid):
    for ext in ("pdb", "cif"):
        p = RAW / f"{pid}.{ext}.gz"
        if p.exists() and p.stat().st_size > 500:
            return pid, p
    for ext in ("pdb", "cif"):
        try:
            with urllib.request.urlopen(
                    f"https://files.rcsb.org/download/{pid}.{ext}.gz", timeout=60) as r:
                data = r.read()
            if len(data) > 500:
                p = RAW / f"{pid}.{ext}.gz"; p.write_bytes(data); return pid, p
        except Exception:
            continue
    return pid, None


t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as ex:
    got = list(ex.map(fetch, needed))
paths = {p: f for p, f in got if f is not None}
print(f"fetched {len(paths):,}/{len(needed):,} in {(time.time()-t0)/60:.1f} min")

## Cell 2: Heavy-atom clouds with per-chain alignment verification (identical to Phase 6b)

In [ ]:
from joblib import Parallel, delayed

AA3 = {"ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE","LEU","LYS",
       "MET","PHE","PRO","SER","THR","TRP","TYR","VAL","MSE"}

want = {}
for i, d in enumerate(dataset):
    want.setdefault(d.pdb_id, []).append((i, d.chain_id, int(d.num_nodes)))


def atoms_for(pid, path, wanted):
    """-> [(dataset_index, [per-residue heavy atoms], (N,3) CA)]"""
    try:
        parser = MMCIFParser(QUIET=True) if ".cif" in path.name else PDBParser(QUIET=True, PERMISSIVE=True)
        with gzip.open(path, "rt") as fh:
            model = next(parser.get_structure(pid, fh).get_models())
    except Exception:
        return []
    out = []
    for idx, cid, n_expect in wanted:
        try:
            ch = model[cid]
        except KeyError:
            continue
        residues = [r for r in ch if r.get_resname().strip().upper() in AA3 and "CA" in r]
        if len(residues) != n_expect:
            continue
        per_res, cas = [], []
        for res in residues:
            xyz = [a.coord for a in res.get_unpacked_list() if a.element not in ("H", "D")]
            per_res.append(np.asarray(xyz, dtype=np.float32).reshape(-1, 3))
            cas.append(np.asarray(res["CA"].coord, dtype=np.float32))   # by NAME
        out.append((idx, per_res, np.stack(cas)))
    return out


t0 = time.time()
batches = Parallel(n_jobs=max(1, os.cpu_count() or 2), verbose=1)(
    delayed(atoms_for)(pid, paths[pid], want[pid]) for pid in sorted(want) if pid in paths)
found = {idx: (pr, ca) for b in batches for idx, pr, ca in b}
print(f"\nparsed in {(time.time()-t0)/60:.1f} min | chains recovered {len(found):,}/{len(dataset):,}")


def geometry_ok(d, ca, tol=1e-2, probe=60):
    """Pairwise CA distances must match the stored graph -> residue ORDER is right."""
    k = min(probe, len(ca))
    p1 = d.pos.numpy()[:k]; p2 = ca[:k]
    a = np.linalg.norm(p1[:, None] - p1[None], axis=-1)
    b = np.linalg.norm(p2[:, None] - p2[None], axis=-1)
    return float(np.abs(a - b).max()) < tol


aligned, bad_geom = [], 0
for i, d in enumerate(dataset):
    if i not in found:
        continue
    pr, ca = found[i]
    if not geometry_ok(d, ca):
        bad_geom += 1
        continue
    aligned.append((i, pr, ca))

print(f"geometry-verified chains: {len(aligned):,} "
      f"(dropped {len(dataset)-len(aligned):,}: {bad_geom} geometry, "
      f"{len(dataset)-len(found)} unparsed)")
assert len(aligned) > 0.8 * len(dataset), "too many chains lost -- investigate before proceeding"

## Cell 3: Featuriser (byte-for-byte the Phase 2b/6b definition)

In [ ]:
MAX_HOM_DIM = 2
BETTI_RADII = (2.0, 3.0, 5.0)
NBHD_RADIUS = 12.0
MAX_NBHD_ATOMS = 300
FEATS_PER_DIM = 6 + 2 + len(BETTI_RADII)
TOPO_DIM = FEATS_PER_DIM * (MAX_HOM_DIM + 1) + 2

TOPO_NAMES = [f"H{d}_{s}" for d in range(MAX_HOM_DIM + 1)
              for s in ["n_bars","total_pers","max_pers","mean_pers","std_pers","entropy",
                        "mean_birth","mean_death", *[f"betti_r{r:g}" for r in BETTI_RADII]]] \
             + ["nbhd_n_atoms", "nbhd_gyration"]
BURIAL_NAMES = ["nbhd_n_atoms", "nbhd_gyration"]


def _diagrams(pts):
    pts = np.ascontiguousarray(pts, dtype=np.float64)
    st, squared = None, True
    if len(pts) >= 5:
        try:
            st = gudhi.AlphaComplex(points=pts).create_simplex_tree()
        except Exception:
            st = None
    if st is None:
        st = gudhi.RipsComplex(points=pts, max_edge_length=float(NBHD_RADIUS)) \
                  .create_simplex_tree(max_dimension=MAX_HOM_DIM + 1)
        squared = False
    st.compute_persistence(homology_coeff_field=2, persistence_dim_max=True)
    out = {}
    for d in range(MAX_HOM_DIM + 1):
        b = np.asarray(st.persistence_intervals_in_dimension(d), dtype=np.float64).reshape(-1, 2)
        out[d] = np.sqrt(np.maximum(b, 0.0)) if (squared and len(b)) else b   # alpha -> angstrom
    return out


def _vectorize(dgms, pts):
    feats = []
    for d in range(MAX_HOM_DIM + 1):
        bars = dgms.get(d, np.zeros((0, 2)))
        fin = bars[np.isfinite(bars).all(axis=1)] if len(bars) else bars.reshape(-1, 2)
        if len(fin) == 0:
            feats.extend([0.0] * FEATS_PER_DIM); continue
        b, dth = fin[:, 0], fin[:, 1]
        pers = np.maximum(dth - b, 0.0); tot = float(pers.sum())
        if tot > 0:
            p = pers / tot; nz = p[p > 0]; ent = float(-(nz * np.log(nz)).sum())
        else:
            ent = 0.0
        feats.extend([float(len(fin)), tot, float(pers.max()), float(pers.mean()),
                      float(pers.std()), ent, float(b.mean()), float(dth.mean()),
                      *[float(((b <= r) & (dth > r)).sum()) for r in BETTI_RADII]])
    c = pts - pts.mean(0, keepdims=True)
    feats.extend([float(len(pts)), float(np.sqrt((c ** 2).sum(1).mean())) if len(pts) else 0.0])
    return np.asarray(feats, dtype=np.float32)


def topo_for_chain(ca_xyz, per_res):
    atoms = np.concatenate([a for a in per_res if len(a)], axis=0)
    tree = cKDTree(atoms)
    out = np.zeros((len(ca_xyz), TOPO_DIM), np.float32)
    for i, ca in enumerate(ca_xyz):
        idx = tree.query_ball_point(ca, NBHD_RADIUS)
        if len(idx) > MAX_NBHD_ATOMS:
            _, idx = tree.query(ca, k=MAX_NBHD_ATOMS)
        pts = atoms[np.asarray(idx, dtype=int)]
        if len(pts) >= 4:
            out[i] = _vectorize(_diagrams(pts), pts)
    return out

## Cell 4: Compute (the long one)

In [ ]:
n_res_total = sum(len(ca) for _, _, ca in aligned)
print(f"computing persistent homology on {n_res_total:,} residues "
      f"({os.cpu_count()} workers, ~6.4 ms/residue -> ETA "
      f"{n_res_total * 0.0064 / 60:.0f} min)")

t0 = time.time()
topo_all = Parallel(n_jobs=max(1, os.cpu_count() or 2), verbose=5)(
    delayed(topo_for_chain)(ca, pr) for _, pr, ca in aligned)
el = time.time() - t0
print(f"\ndone in {el/60:.1f} min ({1000*el/max(n_res_total,1):.2f} ms/residue)")

## Cell 5: Fuse, validate, save (new output filename)

In [ ]:
from sklearn.metrics import roc_auc_score

topo_cat = np.concatenate(topo_all, axis=0)
y_cat = np.concatenate([dataset[i].y.numpy() for i, _, _ in aligned])
print(f"topo matrix {topo_cat.shape} | positives {int(y_cat.sum()):,} "
      f"({100*y_cat.mean():.2f}%)")

keep = topo_cat.std(axis=0) > 1e-9
dropped = [TOPO_NAMES[i] for i in range(TOPO_DIM) if not keep[i]]
topo = topo_cat[:, keep]
names = [TOPO_NAMES[i] for i in range(TOPO_DIM) if keep[i]]
print(f"pruned {len(dropped)} constant column(s): {dropped}")

# --- rotation invariance spot-check ------------------------------------------
g = torch.Generator().manual_seed(0)
Q, _ = torch.linalg.qr(torch.randn(3, 3, generator=g))
Q = (Q * torch.sign(torch.det(Q))).numpy().astype(np.float32)
shift = np.array([31.7, -12.4, 88.1], dtype=np.float32)
_, pr0, ca0 = aligned[0]
sub, cas = pr0[:40], ca0[:40]
f_ref = topo_for_chain(cas, sub)
f_rot = topo_for_chain(cas @ Q.T + shift, [a @ Q.T + shift for a in sub])
rel = float((np.abs(f_ref - f_rot) / np.maximum(np.abs(f_ref).max(0), 1e-6)).max())
print(f"rotation invariance: {rel:.2e}")
assert rel < 1e-3

aucs = np.array([0.5 if topo[:, j].std() < 1e-9 else roc_auc_score(y_cat, topo[:, j])
                 for j in range(topo.shape[1])])
order = np.argsort(-np.abs(aucs - 0.5))
print("\ntop single-feature AUC (n=%d residues):" % len(y_cat))
for j in order[:8]:
    print(f"   {names[j]:<22} {aucs[j]:.3f}")

mu, sd = topo.mean(0), topo.std(0); sd[sd < 1e-6] = 1.0
out_ds = []
for (i, _, _), t in zip(aligned, topo_all):
    d = dataset[i]
    t = t[:, keep]
    d.x_topo = torch.from_numpy(t.astype(np.float32))
    d.x = torch.cat([d.x, torch.from_numpy(((t - mu) / sd).astype(np.float32))], dim=1)
    assert torch.isfinite(d.x).all()
    out_ds.append(d)

out = Path("/kaggle/working/processed"); out.mkdir(parents=True, exist_ok=True)
p = out / "phase29_graphs.pt"
torch.save({"dataset": out_ds, "node_feat_dim": int(out_ds[0].x.size(1)),
            "edge_feat_dim": int(out_ds[0].edge_attr.size(1)),
            "topo_dim": int(topo.shape[1]), "topo_names": names,
            "burial_names": BURIAL_NAMES, "dropped_names": dropped,
            "topo_mean": mu, "topo_std": sd, "topo_auc": aucs,
            "pos_weight": blob["pos_weight"], "aa_alphabet": blob["aa_alphabet"]}, p)
print(f"\nchains {len(out_ds):,} | node dim {out_ds[0].x.size(1)}")
print(f"saved -> {p} ({p.stat().st_size/1e6:.0f} MB)")

---

`phase29_graphs.pt` -- same schema as `phase6b_graphs.pt` (dataset with
fused x_topo/x, topo_dim, topo_names, burial_names, topo_mean/std, topo_auc,
pos_weight, aa_alphabet), just the bigger pool. Next: Phase 30 (dihedrals +
ESM2, mirrors Phase 9/10) to reach the final trainable schema.